In [1]:
import sys
import os

parent_dir = os.path.abspath(os.path.join(".."))
sys.path.append(parent_dir)

In [ ]:
import torch
import torch.optim as optim
from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import ProjectedGradientDescent
from art.defences.trainer import AdversarialTrainer
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from utils.cnn_architecture import SimpleCNN
from utils.data_transformation import transform
import numpy as np

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

### Load the data

Train dataset

In [ ]:
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

In [ ]:
train_dataset = datasets.CIFAR10(root='../data', train=True, download=True, transform=transform_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [ ]:
def dataloader_to_numpy(dataloader):
    # Convert the DataLoader to numpy arrays
    x_list = []
    y_list = []
    for inputs, labels in dataloader:
        x_list.append(inputs.numpy())
        y_list.append(labels.numpy())
    
    x_np = np.concatenate(x_list, axis=0)  # Stack all the batches
    y_np = np.concatenate(y_list, axis=0)
    return x_np, y_np


x_train, y_train = dataloader_to_numpy(train_loader)

Test dataset

In [ ]:
test_dataset = datasets.CIFAR10(root='../data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

### Load the model

In [ ]:
model = torch.load('../models/cifar10_basic_model.pth')
model.to(device)

In [ ]:
model.train()  # Switch to train mode

### Train the new model

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [ ]:
classifier = PyTorchClassifier(
    model=model,
    loss=criterion,
    optimizer=optimizer,
    input_shape=(3, 32, 32),
    nb_classes=10
)

Attack implementation

In [ ]:
attack = ProjectedGradientDescent(
    estimator=classifier,
    eps=0.3,
    eps_step=0.05,
    max_iter=10, 
    num_random_init=2
)

In [ ]:
trainer = AdversarialTrainer(classifier, attacks=attack, ratio=0.5)

In [2]:
trainer.fit(x_train, y_train, nb_epochs=10)

/home/adavodet/PROJECTS/cifar_classification/cifar_env/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda
Files already downloaded and verified


/tmp/ipykernel_4026690/1524890371.py:28: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load('../models/cifar10_basic_model.pth')
Adversarial training epochs: 1

### Model evaluation

In [ ]:
model.eval()  # Switch to evaluation mode

In [ ]:
correct = 0
total = 0

In [3]:
with torch.no_grad():
    for data in test_loader:
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f'Accuracy on clean test data: {100 * correct / total:.2f}%')


Files already downloaded and verified
Accuracy on clean test data: 52.00%


### Save the model

In [4]:
torch.save(model, '../models/adversarially_trained_model.pth')